In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.2")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': " I am a large language model trained by Mistral AI. I am designed to generate human-like text based on the input I receive. I don't have the ability to have experiences or emotions, but I can process text and generate responses based on patterns and information I have learned during my training."}]}]

In [3]:
import analogy_dataset as ad

In [4]:
# Get dataset information
info = ad.get_dataset_info()
print(f"Total records: {info['total_records']}")
print(f"Access rules: {info['data_access_rules']}")

Total records: 405
Access rules: {'full_access_columns': ['All_Correct_Answers'], 'row_by_row_only': ['Index', 'Part1', 'Part2', 'Part3', 'Part4', 'Formatted_Question', 'Question_Type', 'Option_1', 'Option_2', 'Option_3', 'Option_4', 'Option_5'], 'access_method': 'Use get_iterator() for row-by-row access'}


In [5]:
# Access answers in full (for evaluation)
answers = ad.get_answers()
print(f"Total answers: {len(answers)}")

Total answers: 405


In [6]:
# Row-by-row access to questions and other data
iterator = ad.get_iterator(random_state=123)
row = iterator.get_next_row()
if row:
    print(f"Question: {row['Formatted_Question']}")
    print(f"Answer: {row['All_Correct_Answers']}")

Question: चाकू : कटना :: बन्दूक : ?
Answer: गोली मारना


In [7]:
#import analogy_dataset as ad

# Get comprehensive dataset information
info = ad.get_dataset_info()
print(f"Total records: {info['total_records']}")
print(f"Columns: {info['columns']}")
print(f"Question types: {info['question_types']}")

# Get statistics
stats = ad.get_statistics()
print(f"Total analogies: {stats['total_analogies']}")
print(f"Question distribution: {stats['question_type_distribution']}")

Total records: 405
Columns: ['Index', 'Part1', 'Part2', 'Part3', 'Part4', 'Formatted Question', 'Question Type_new', 'All_Correct_Answers', 'Option 1', 'Option 2', 'Option 3', 'Option 4', 'Option 5']
Question types: {1: 331, 2: 74}
Total analogies: 405
Question distribution: {1: 331, 2: 74}


In [9]:
# Create an iterator
iterator = ad.get_iterator(random_state=123)

# Process data row by row
while iterator.has_next():
    row = iterator.get_next_row()
    print(f"Q: {row['Formatted_Question']}")
    print(f"A: {row['All_Correct_Answers']}")
    print("---")
    
    # Process only first 3 for example
    if iterator.get_current_position() >= 1:
        break

# Reset iterator to start over
iterator.reset()

Q: चाकू : कटना :: बन्दूक : ?
A: गोली मारना
---


In [10]:
# Get iterator for specific question type
type1_iterator = ad.get_iterator(question_type=1, random_state=42)
type2_iterator = ad.get_iterator(question_type=2, random_state=42)

print(f"Type 1 questions: {type1_iterator.get_total_rows()}")
print(f"Type 2 questions: {type2_iterator.get_total_rows()}")

Type 1 questions: 331
Type 2 questions: 74


In [11]:
# Get all answers for evaluation
answers = ad.get_answers()
print(f"Total answers: {len(answers)}")

# Search within answers
cow_answers = ad.search_answers("गाय")
print(f"Answers containing 'गाय': {len(cow_answers)}")

# Export answers for evaluation
ad.export_answers_to_csv("all_answers.csv")

Total answers: 405
Answers containing 'गाय': 2
Answers exported to all_answers.csv


In [15]:
import analogy_dataset as ad
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
from tqdm import tqdm

# ---------------------------------------------------------
# Step 1: Load Mistral 7B Instruct v0.2
# ---------------------------------------------------------
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ---------------------------------------------------------
# Step 2: Prompt Builder
# ---------------------------------------------------------
def build_prompt(row):
    return f"""You are a multilingual translator.
Translate this analogy MCQ entirely from Hindi to Tamil.

Question: {row['Part1']} : {row['Part2']} :: {row['Part3']} : ?
Options:
A. {row['Option_1']}
B. {row['Option_2']}
C. {row['Option_3']}
D. {row['Option_4']}
"""

# ---------------------------------------------------------
# Step 3: Translation Function (Mistral chat mode)
# ---------------------------------------------------------
def get_translation(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    output_ids = model.generate(
        encoded,
        max_new_tokens=512,
        do_sample=False
    )

    # remove prompt tokens → keep only generated part
    generated_text = tokenizer.decode(
        output_ids[0][encoded.shape[-1]:],
        skip_special_tokens=True
    )

    return generated_text.strip()

# ---------------------------------------------------------
# Step 4: Load Analogy Dataset
# ---------------------------------------------------------
all_rows = []
iterator = ad.get_iterator()

while True:
    row = iterator.get_next_row()
    if not row:
        break
    all_rows.append(row)

batch_size = 20
total = len(all_rows)

# choose which batch to run:
batch_index = 0

start = batch_index * batch_size
end = min(start + batch_size, total)
batch_rows = all_rows[start:end]

translated_data = []

for row in tqdm(batch_rows):
    prompt = build_prompt(row)

    try:
        tamil = get_translation(prompt)
    except Exception as e:
        tamil = f"ERROR: {e}"
        print("Error at:", row["Formatted_Question"], e)

    translated_data.append({
        "Formatted_Question": row["Formatted_Question"],
        "Answer_Hindi": row["All_Correct_Answers"],
        "Tamil_Translated": tamil
    })

# ---------------------------------------------------------
# Step 5: Save Translated Batch
# ---------------------------------------------------------
df = pd.DataFrame(translated_data)
df.to_csv(f"tats_batch_{batch_index}_mistral7b.csv", index=False)

print(f"Batch {batch_index} saved successfully!")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attentio

Batch 0 saved successfully!


In [16]:
!nvidia-smi

Fri Dec  5 21:33:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.07             Driver Version: 570.133.07     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


|   0  NVIDIA A40                     Off |   00000000:52:00.0 Off |                    0 |
|  0%   36C    P0             77W /  300W |   42108MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
                                                                                         
+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    0   N/A  N/A            1937      G   /usr/lib/xorg/Xorg                     